## Loading the dataset

### imports

In [2]:
from datasets import load_dataset
import pandas as pd
import re
from tokenizers import Tokenizer, models, pre_tokenizers, trainers
from pathlib import Path
import json
import pickle

### download WikiText-2

In [3]:
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
dataset.save_to_disk("../data/raw/wikitext2_raw")
print(dataset)

Saving the dataset (0/1 shards):   0%|          | 0/4358 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/36718 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3760 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})


---

### Inspecting

In [114]:
for split_name in dataset.keys():
    print(split_name, "examples:", len(dataset[split_name]))

test examples: 4358
train examples: 36718
validation examples: 3760


In [115]:
for i in range(30): # first 20 sentences in train
    print(repr(dataset["train"][i]["text"]))

''
' = Valkyria Chronicles III = \n'
''
' Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . \n'
" The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgiving 

In [116]:
def inspect_split(split):
    texts = [example["text"] for example in split]
    
    return {
        "total_lines": len(texts),
        "empty_lines": sum(not text.strip() for text in texts),
        "non_empty_lines": sum(bool(text.strip()) for text in texts),
        "total_characters": sum(len(text) for text in texts),
    }
    
original_stats = pd.DataFrame({name: inspect_split(dataset[name]) for name in dataset.keys()}).T

original_stats

,total_lines,empty_lines,non_empty_lines,total_characters
test,4358,1467,2891,1285622
train,36718,12951,23767,10892990
validation,3760,1299,2461,1142150


In [117]:
def inspect_artifacts(split):
    texts = [example["text"] for example in split]

    return {
        "heading_lines": sum(bool(re.fullmatch(r"=+.*=+", text.strip())) for text in texts if text.strip()),
        "lines_with_at_dash_at": sum("@-@" in text for text in texts),
        "lines_with_at_comma_at": sum("@,@" in text for text in texts),
        "lines_with_at_dot_at": sum("@.@" in text for text in texts),
    }

artifact_stats = pd.DataFrame({split_name: inspect_artifacts(dataset[split_name])for split_name in dataset.keys()}).T

artifact_stats

,heading_lines,lines_with_at_dash_at,lines_with_at_comma_at,lines_with_at_dot_at
test,706,1023,168,204
train,6211,8198,1331,1346
validation,620,891,200,117


In [118]:
for i, example in enumerate(dataset["train"]):
    text = example["text"]

    if text.strip() and (re.fullmatch(r"=+.*=+", text.strip()) or "@-@" in text or "@,@" in text or "@.@" in text): # just to see sentences with this properties
        print(repr(text))

    if i > 200:
        break

' = Valkyria Chronicles III = \n'
' Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . \n'
' = = Gameplay = = \n'
" As with previous Valkyira Chronicles games , Valkyria Chronicles III is a tactical role @-@ playing game where players take control of a military unit and take part in missions against enemy forces . Stories are told through comic 

---

### Cleaning 

In [119]:
MARKER_RE  = re.compile(r"\s*@([-,.])@\s*") # " @-@ ", " @,@ ", " @.@ "
HEADING_RE = re.compile(r"=( =)* .+ =( =)*") # "= Title =", "= = Sub = ="

def clean_line(text, restore_markers= True):
   text = text.strip()
   if not text or HEADING_RE.fullmatch(text): # if the text is empty or it matches the heading we return none
       return None
   if restore_markers:
       text = MARKER_RE.sub(r"\1", text) # so putting the pieces back together
       
   text = re.sub(r"\s+", " ", text).strip()
   return text or None 

# see if it works
examples = [" = = Gameplay = = ", "role @-@ playing video game", "102 @,@ 779 units", "1 @.@ 38 goals against average","   ",]

for example in examples:
    print("Before:", repr(example))
    print("After: ", repr(clean_line(example)))
    print()

Before: ' = = Gameplay = = '
After:  None

Before: 'role @-@ playing video game'
After:  'role-playing video game'

Before: '102 @,@ 779 units'
After:  '102,779 units'

Before: '1 @.@ 38 goals against average'
After:  '1.38 goals against average'

Before: '   '
After:  None



In [120]:
# applying on the data
cleaned_lines = {}

for split_name in dataset.keys():
    cleaned_lines[split_name] = []
    
    for example in dataset[split_name]:
        cleaned = clean_line(example["text"])  
        if cleaned is not None:
            cleaned_lines[split_name].append(cleaned)
    
    print(f"{split_name}:")
    print("Original lines:", len(dataset[split_name]))
    print("Cleaned lines:", len(cleaned_lines[split_name]))
    print()
    

test:
Original lines: 4358
Cleaned lines: 2185

train:
Original lines: 36718
Cleaned lines: 17556

validation:
Original lines: 3760
Cleaned lines: 1841



In [121]:
for i in range(5): # to see five cleaned sentences
    print(f"Example {i + 1}:")
    print(cleaned_lines["train"][i])
    print()

Example 1:
Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role-playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real-time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " .

Example 2:
The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgiving for series newcomers . Character 

In [122]:
heading_examples = [
    "= Valkyria Chronicles =",
    "= = Gameplay = =",
]

for heading in heading_examples:
    print(repr(heading), bool(HEADING_RE.fullmatch(heading)))

'= Valkyria Chronicles =' True
'= = Gameplay = =' True


In [123]:
# to check if artifacts are cleaned or what
remaining_artifacts = {}

for split_name, lines in cleaned_lines.items():
    remaining_artifacts[split_name] = {
        "lines": len(lines),
        "lines_with_at_dash_at": sum("@-@" in line for line in lines),
        "lines_with_at_comma_at": sum("@,@" in line for line in lines),
        "lines_with_at_dot_at": sum("@.@" in line for line in lines),
        "heading_only_lines": sum(bool(re.fullmatch(r"=+.*=+", line.strip())) for line in lines),
    }

pd.DataFrame(remaining_artifacts).T

,lines,lines_with_at_dash_at,lines_with_at_comma_at,lines_with_at_dot_at,heading_only_lines
test,2185,0,0,0,0
train,17556,0,0,0,0
validation,1841,0,0,0,0


so all good

---  

### Tokenization

In [ ]:

def tokenize_line(text):
    # split a line into words using whitespace
    return text.split()


examples = ["The game was released in 2013.", "role - playing video game", "The score was 102,779."]

for example in examples:
    print("Text:", example)
    print("Tokens:", tokenize_line(example))
    print()

Text: The game was released in 2013.
Tokens: ['The', 'game', 'was', 'released', 'in', '2013.']

Text: role - playing video game
Tokens: ['role', '-', 'playing', 'video', 'game']

Text: The score was 102,779.
Tokens: ['The', 'score', 'was', '102,779.']



In [125]:
# then applying it on the data
tokenized_lines = {}

for split_name, lines in cleaned_lines.items():
    tokenized_lines[split_name] = [tokenize_line(line) for line in lines]
    
    total_tokens = sum(len(tokens) for tokens in tokenized_lines[split_name])
    
    print(f"{split_name}:")
    print("Lines:", len(tokenized_lines[split_name]))
    print("Tokens:", total_tokens)
    print()


test:
Lines: 2185
Tokens: 229880

train:
Lines: 17556
Tokens: 1961891

validation:
Lines: 1841
Tokens: 204347



In [126]:
# and now see the tokens
for i in range(3):
    print(f"Example {i + 1}")
    print("Original:", cleaned_lines["train"][i])
    print("Tokens:", tokenized_lines["train"][i])
    print()

Example 1
Original: Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role-playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real-time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " .
Tokens: ['Senjō', 'no', 'Valkyria', '3', ':', 'Unrecorded', 'Chronicles', '(', 'Japanese', ':', '戦場のヴァルキュリア3', ',', 'lit', '.', 'Valkyria', 'of', 'the', 'Battlefield', '3', ')', ',', 'commonly', 'referred', 'to', 'as', 'Valkyria', 'Chronicles', 'III', 'outside', 'Japan', ',', 'is', '

In [127]:
## checking the vocabulary size
for split_name, lines in tokenized_lines.items():
    unique_tokens = {token for tokens in lines for token in tokens}
    print(f"{split_name}:")
    print("Unique tokens:", len(unique_tokens))
    print()

test:
Unique tokens: 22272

train:
Unique tokens: 85297

validation:
Unique tokens: 20897



---

### Vocabualry

but I rather use hugging face models and tokenizer for tokenization, just to try out sth different from the assignment

In [ ]:
# special tokens used by our models
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
BOS_TOKEN = "<bos>"
EOS_TOKEN = "<eos>"

special_tokens = [PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN]

hf_tokenizer = Tokenizer(models.WordLevel(unk_token=UNK_TOKEN)) # word-level tokenizer
hf_tokenizer.pre_tokenizer = pre_tokenizers.WhitespaceSplit() ## splitting text at whitespace
# learn the vocabulary from training data only
trainer = trainers.WordLevelTrainer(vocab_size= 200_000, special_tokens=special_tokens, min_frequency=2)
hf_tokenizer.train_from_iterator(cleaned_lines["train"], trainer=trainer)

vocab = [hf_tokenizer.id_to_token(i) for i in range(hf_tokenizer.get_vocab_size())]

word_to_id = {word: idx for idx, word in enumerate(vocab)}
id_to_word = {idx: word for word, idx in word_to_id.items()}

print("Vocabulary size:", len(vocab))
print("Special token IDs:", {token: word_to_id[token] for token in special_tokens})

Vocabulary size: 85301
Special token IDs: {'<pad>': 0, '<unk>': 1, '<bos>': 2, '<eos>': 3}


In [129]:
encoded_lines = {}

bos_id = word_to_id[BOS_TOKEN]
eos_id = word_to_id[EOS_TOKEN]
unk_id = word_to_id[UNK_TOKEN]

for split_name, lines in cleaned_lines.items():
    encoded_lines[split_name] = []
    for line in lines:
        token_ids = hf_tokenizer.encode(line).ids #converting words into token IDs.
        # and then replace the sequence boundaries
        token_ids = [bos_id] + token_ids + [eos_id]
        encoded_lines[split_name].append(token_ids)

    unknown_count = sum(token_id == unk_id for sequence in encoded_lines[split_name] for token_id in sequence)

    print(f"{split_name}:")
    print("Sequences:", len(encoded_lines[split_name]))
    print("Unknown tokens:", unknown_count)
    print()


test:
Sequences: 2185
Unknown tokens: 10921

train:
Sequences: 17556
Unknown tokens: 0

validation:
Sequences: 1841
Unknown tokens: 8303



In [130]:
print(hf_tokenizer.encode("playing").tokens)
print(hf_tokenizer.encode("unbelievable").tokens)
print(hf_tokenizer.encode("102,779").tokens)

['playing']
['unbelievable']
['102,779']


---

### Saving files

In [131]:
PROJECT_DIR = Path.cwd().parent
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [132]:
with open(PROCESSED_DIR / "vocab.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "vocab" : vocab,
            "word_to_id": word_to_id,
            "special_tokens": special_tokens,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )
    
with open(PROCESSED_DIR/"encoded_lines.pkl", "wb") as f:
    pickle.dump(encoded_lines, f)
    
    
hf_tokenizer.save(str(PROCESSED_DIR / "tokenizer.json"))